# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Print available record sets, their @id and fields
print("Available Record Sets (@id):")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']}: {rs.get('name','(no name)')}")
    print("  Fields/Columns:")
    for fld in rs['fields']:
        # For each field, print @id and name if present
        name_str = f" ({fld.get('name','')})" if 'name' in fld else ""
        print(f"    - {fld['@id']}{name_str}")
    print()

# Let's list them in variables for later referencing
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set])} records for Record Set: {record_set}")
    print(f"Fields: {list(dataframes[record_set].columns)}\n")

# For demonstration, select the first record set
main_record_set_id = record_sets[0]
print(f"Columns in DataFrame for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Try to automatically pick candidate numeric and grouping fields by @id (fall back if not found)
import numpy as np

df = dataframes[main_record_set_id].copy()
# Select a likely numeric field by searching for columns with float/int dtype
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Attempt to convert numericish columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field = col
            break
        except:
            continue
    else:
        raise RuntimeError('No numeric fields found for EDA!')

print(f"Using '{numeric_field}' as the numeric field for analysis.")
# Use median as a sensible threshold for demonstration
if df[numeric_field].dtype==object:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].median()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to choose a non-numeric/grouping field
categorical_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = categorical_candidates[0] if categorical_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (showing first 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field is not None:
    plt.figure(figsize=(10,5))
    order = df[group_field].value_counts().index[:10]
    sns.boxplot(x=group_field, y=numeric_field, data=df, order=order)
    plt.title(f"{numeric_field} by {group_field} (Top 10 Groups)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, inspecting, and analyzing the FAIR\(^2\) mass spectrometry dataset using Croissant metadata with `mlcroissant`.

**Key findings:**
- Dataset fields and record sets are referenced by global `@id` for reproducibility.
- We loaded data directly from Croissant metadata and performed basic exploratory data analysis, including filtering, normalization, and grouping.
- Basic distribution and group-wise visualizations were created for quick insights.

**Next steps:**
- Leverage further domain knowledge to focus on key proteins or variables of biological interest.
- Explore data semantics (e.g., peptide sequences, modifications) for more advanced analyses.
- Take advantage of Croissant's provenance and FAIR metadata for transparent and reusable ML workflows.